# 01 · TIFF to Zarr (Chunked)
Reads Opera Phenix `.tiff` output for experiment **BPP0050**, parses Harmony XML metadata
via `macrohet.dataio`, and tiles multi-position acquisitions into OME-Zarr mosaics.
A feathered alpha mask is applied at tile boundaries to reduce stitching artefacts.

Two tiling strategies are implemented:
- **Batch**: loads all timepoints into memory before writing (used for all completed acquisitions)
- **Chunked**: writes one timepoint at a time directly to zarr (used for Live-2 which was too large for batch)

In [ ]:
import os
import glob
import shutil
import pandas as pd
import numpy as np
import zarr
import dask.array
import numcodecs
import xml.etree.ElementTree as ET
from pathlib import Path
from scipy import stats
from tqdm.auto import tqdm
from skimage.io import imread
from macrohet import dataio

## Helper functions

In [ ]:
def create_alpha_mask(height, width, feather=0.1):
    """
    Create a 2D alpha mask of shape (height, width).
    The mask is 1 in the interior and linearly ramps down to 0
    at each edge over a fraction `feather` of the image dimensions.
    Enables smooth feathered tile blending during mosaic stitching.

    Args:
        height (int): Tile height in pixels.
        width (int): Tile width in pixels.
        feather (float): Fraction of the shorter dimension over which to ramp. Default 0.1.

    Returns:
        np.ndarray: Float32 alpha mask of shape (height, width).
    """
    y = np.arange(height)
    x = np.arange(width)
    feather_pixels = feather * min(height, width)

    ramp_y = np.minimum(y, (height - 1) - y)
    ramp_x = np.minimum(x, (width  - 1) - x)

    dist_from_edge = np.minimum(ramp_y[:, None], ramp_x[None, :])
    alpha = np.clip(dist_from_edge / feather_pixels, 0, 1)
    return alpha.astype(np.float32)


def load_assay_layout(xml_file_name):
    """
    Parse a Harmony assay layout XML file into a DataFrame indexed by well position.

    Args:
        xml_file_name (str): Full path to the assay layout XML file.

    Returns:
        pd.DataFrame: Indexed by position string e.g. "(3, 4)", columns are layer names.
                      Returns empty DataFrame on error.
    """
    df_assay_layout = pd.DataFrame()
    if not os.path.exists(xml_file_name):
        print(f"Error: '{xml_file_name}' not found.")
        return df_assay_layout
    try:
        tree = ET.parse(xml_file_name)
        root = tree.getroot()
        namespaces = {'ns': 'http://www.perkinelmer.com/PEHH/HarmonyV5'}
        plate_data = {}

        for layer in root.findall('ns:Layer', namespaces):
            layer_name_el = layer.find('ns:Name', namespaces)
            if layer_name_el is None or layer_name_el.text is None:
                print("Warning: Layer without a name, skipping.")
                continue
            layer_name = layer_name_el.text.strip()

            for well in layer.findall('ns:Well', namespaces):
                row_el = well.find('ns:Row', namespaces)
                col_el = well.find('ns:Col', namespaces)
                val_el = well.find('ns:Value', namespaces)
                if row_el is None or col_el is None:
                    continue
                try:
                    row = int(row_el.text.strip())
                    col = int(col_el.text.strip())
                except ValueError:
                    continue
                value = val_el.text.strip() if val_el is not None and val_el.text else pd.NA
                plate_data.setdefault((row, col), {})[layer_name] = value

        if plate_data:
            df_assay_layout = pd.DataFrame.from_dict(plate_data, orient='index')
            df_assay_layout.index.names = ['Row', 'Col']
            df_assay_layout = df_assay_layout.sort_index().dropna(how='all')
            df_assay_layout['position'] = df_assay_layout.index.map(lambda rc: f"({rc[0]}, {rc[1]})")
            df_assay_layout = df_assay_layout.set_index('position').drop(columns=['Row', 'Col'])
        else:
            print("No data extracted from XML.")

    except ET.ParseError as e:
        print(f"XML parse error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

    return df_assay_layout

## 1 · Create acquisition subdirectories
Organises raw measurement folders under a common `acquisition/` subdirectory.

In [ ]:
base_dir = '/mnt/DATA3/BPP0050/'
parent_directories = glob.glob(os.path.join(base_dir, '*/'))
ACQUISITIONS_DIR_NAME = 'acquisition'

for parent_dir in parent_directories:
    parent_dir = os.path.normpath(parent_dir)
    if not os.path.isdir(parent_dir):
        print(f"Warning: {parent_dir} not found, skipping.")
        continue
    acquisitions_path = os.path.join(parent_dir, ACQUISITIONS_DIR_NAME)
    os.makedirs(acquisitions_path, exist_ok=True)

    # Move existing subdirectories (Images, Assaylayout etc.) into acquisition/
    items = [
        d for d in os.listdir(parent_dir)
        if os.path.isdir(os.path.join(parent_dir, d)) and d != ACQUISITIONS_DIR_NAME
    ]
    for subdir_name in items:
        src = os.path.join(parent_dir, subdir_name)
        dst = os.path.join(acquisitions_path, subdir_name)
        try:
            shutil.move(src, dst)
            print(f"  Moved '{src}' -> '{dst}'")
        except shutil.Error as e:
            print(f"  Error moving '{src}': {e}")

print("Subdirectory organisation complete.")

## 2 · Batch tiling: all acquisitions
For each acquisition directory, reads Harmony metadata, iterates over positions
and builds a (T, C, Z, Y, X) mosaic in memory before writing to zarr.
Tile boundaries are blended using the feathered alpha mask.
DPC (Phase Contrast) slices are distributed uniformly across Z.
OMERO and multiscales metadata are written to the zarr root `.zattrs` after tiling.

In [ ]:
base_dirs = glob.glob(os.path.join('/mnt/DATA3/BPP0050/', '*/'))

for base_dir in tqdm(base_dirs, total=len(base_dirs)):
    base_dir_ID = base_dir.split('/')[-2]

    metadata_fn = os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
    metadata = dataio.read_harmony_metadata(metadata_fn)
    metadata = metadata.apply(pd.to_numeric, errors='ignore')

    # Invert Y coordinates (cathode ray tube adjustment)
    metadata['PositionY'] = -metadata['PositionY']

    # Align Phase Contrast Z to the minimum non-PC Z position
    min_z = metadata.loc[
        ~metadata['ChannelName'].str.contains('Phase Contrast', case=False, na=False),
        'PositionZ'
    ].min()
    metadata.loc[
        metadata['ChannelName'].str.contains('Phase Contrast', case=False, na=False),
        'PositionZ'
    ] = min_z

    image_dir  = os.path.join(base_dir, 'acquisition/Images/')
    rows_cols  = metadata[['Row', 'Col']].drop_duplicates().values
    sample_tile_id = metadata.iloc[0]

    assay_layout_fn = glob.glob(os.path.join(base_dir, 'acquisition/Assaylayout/*.xml'))[0]
    assay_layout    = load_assay_layout(assay_layout_fn)

    for row_col in tqdm(rows_cols, desc=f'Tiling {base_dir_ID}', total=len(rows_cols)):
        row, column = int(row_col[0]), int(row_col[1])
        acq_ID = (row, column)

        zarr_output_dir = os.path.join(base_dir, f'acquisition/zarr/{acq_ID}.zarr')
        if os.path.exists(zarr_output_dir):
            print(f"{zarr_output_dir} already exists, skipping")
            continue

        position_metadata = metadata[(metadata['Row'] == row) & (metadata['Col'] == column)]
        timepoint_ids     = sorted(position_metadata['TimepointID'].unique())
        time_volumes      = []

        for timepoint_id in tqdm(timepoint_ids, desc="Time Points", leave=True, position=1):
            time_metadata  = position_metadata[position_metadata['TimepointID'] == timepoint_id]
            channel_ids    = sorted(time_metadata['ChannelID'].unique())
            channel_volumes = []

            for channel_id in tqdm(channel_ids, desc=f"Channels (T={timepoint_id})", leave=False, position=2):
                channel_slice_metadata = time_metadata[time_metadata['ChannelID'] == channel_id]
                z_ids    = sorted(channel_slice_metadata['PositionZ'].unique())
                z_slices = []

                for z_id in z_ids:
                    z_slice_metadata = channel_slice_metadata[channel_slice_metadata['PositionZ'] == z_id]
                    mosaic_size_x = (
                        int((z_slice_metadata['PositionX'].max() - z_slice_metadata['PositionX'].min())
                            / sample_tile_id['ImageResolutionX'])
                        + sample_tile_id['ImageSizeX']
                    )
                    mosaic_size_y = (
                        int((z_slice_metadata['PositionY'].max() - z_slice_metadata['PositionY'].min())
                            / sample_tile_id['ImageResolutionY'])
                        + sample_tile_id['ImageSizeY']
                    )
                    accumulator = np.zeros((mosaic_size_y, mosaic_size_x), dtype=np.float32)
                    weight_map  = np.zeros((mosaic_size_y, mosaic_size_x), dtype=np.float32)

                    for _, tile_id in z_slice_metadata.iterrows():
                        img_path = os.path.join(image_dir, tile_id['URL'])
                        try:
                            tile = imread(img_path).astype(np.float32)
                        except FileNotFoundError:
                            tile = np.zeros((tile_id['ImageSizeY'], tile_id['ImageSizeX']), dtype=np.float32)

                        alpha_mask = create_alpha_mask(tile.shape[0], tile.shape[1], feather=0.1)
                        x_pixel = int((tile_id['PositionX'] - z_slice_metadata['PositionX'].min())
                                      / tile_id['ImageResolutionX'])
                        y_pixel = int((tile_id['PositionY'] - z_slice_metadata['PositionY'].min())
                                      / tile_id['ImageResolutionY'])

                        accumulator[y_pixel:y_pixel+tile.shape[0], x_pixel:x_pixel+tile.shape[1]] += tile * alpha_mask
                        weight_map [y_pixel:y_pixel+tile.shape[0], x_pixel:x_pixel+tile.shape[1]] += alpha_mask

                    final_mosaic = np.zeros_like(accumulator)
                    nonzero = weight_map > 0
                    final_mosaic[nonzero] = accumulator[nonzero] / weight_map[nonzero]
                    mosaic = np.clip(final_mosaic, 0, 65535).astype(np.uint16)

                    # DPC: distribute across all Z slices rather than stacking
                    if "Phase Contrast" in channel_slice_metadata['ChannelName'].iloc[0]:
                        z_slices = [mosaic] * len(metadata['PositionZ'].unique())
                        break
                    else:
                        z_slices.append(mosaic)

                if z_slices:
                    channel_volumes.append(np.stack(z_slices, axis=0))

            if channel_volumes:
                time_volumes.append(np.stack(channel_volumes, axis=0))  # (C, Z, Y, X)

        if time_volumes:
            images = np.stack(time_volumes, axis=0)  # (T, C, Z, Y, X)
            print(f"Final image volume shape: {images.shape}")
            try:
                dask_images = dask.array.from_array(images)
                dask.array.to_zarr(dask_images, zarr_output_dir, component='images')
                print(f"Saved: {zarr_output_dir}")
            except Exception:
                print(f"Zarr output failed: {base_dir_ID} {acq_ID}")

    # --- Build and write OMERO + multiscales metadata ---
    omero_metadata = {}
    average_time_difference_seconds = None

    if 'AbsTime' in metadata.columns and 'TimepointID' in metadata.columns:
        try:
            if not pd.api.types.is_datetime64_any_dtype(metadata['AbsTime']):
                metadata['AbsTime'] = pd.to_datetime(metadata['AbsTime'], format='ISO8601', utc=True)
            timepoint_abs_times = (
                metadata[['TimepointID', 'AbsTime']].drop_duplicates(subset=['TimepointID'])
                .sort_values(by='TimepointID', key=lambda x: pd.to_numeric(x, errors='ignore'))
            )
            time_diffs = timepoint_abs_times['AbsTime'].diff().dt.total_seconds().dropna()
            if not time_diffs.empty:
                average_time_difference_seconds = time_diffs.mean()
                if average_time_difference_seconds > 0:
                    omero_metadata['frameRate'] = 1 / average_time_difference_seconds
        except Exception as e:
            print(f"Warning: framerate calculation failed: {e}")

    channels_data = []
    for _, row in metadata.drop_duplicates(subset=['ChannelID']).iterrows():
        emission_wl   = None
        excitation_wl = None
        if pd.notna(row['MainEmissionWavelength']):
            try:
                emission_wl = float(row['MainEmissionWavelength']) * 1e-9
            except ValueError:
                pass
        if pd.notna(row['MainExcitationWavelength']):
            try:
                excitation_wl = float(row['MainExcitationWavelength']) * 1e-9
            except ValueError:
                pass
        channels_data.append({
            "id":                   int(row['ChannelID']),
            "label":                row['ChannelName'],
            "emissionWaveMeters":   emission_wl,
            "excitationWaveMeters": excitation_wl,
        })
    omero_metadata['channels'] = channels_data

    objective_data = {}
    if 'ObjectiveMagnification' in metadata.columns and pd.notna(metadata['ObjectiveMagnification'].iloc[0]):
        try:
            objective_data['magnification'] = float(metadata['ObjectiveMagnification'].iloc[0])
        except ValueError:
            pass
    if 'ObjectiveNA' in metadata.columns and pd.notna(metadata['ObjectiveNA'].iloc[0]):
        try:
            objective_data['numericalAperture'] = float(metadata['ObjectiveNA'].iloc[0])
        except ValueError:
            pass
    if objective_data:
        omero_metadata['objective'] = objective_data

    multiscales_data = [{
        "name": "images",
        "datasets": [],
        "axes": [
            {"name": "t", "type": "time",    "unit": "second", "spacing": average_time_difference_seconds},
            {"name": "c", "type": "channel"},
            {"name": "z", "type": "space",   "unit": "meter",  "spacing": None},
            {"name": "y", "type": "space",   "unit": "meter",  "pixelResolution": None},
            {"name": "x", "type": "space",   "unit": "meter",  "pixelResolution": None}
        ],
        "type": "image", "version": "0.4"
    }]

    zarr_root_dir = os.path.join(base_dir, 'acquisition/zarr/')
    for filename in os.listdir(zarr_root_dir):
        if filename.endswith('.zarr'):
            multiscales_data[0]['datasets'].append({"path": f"{filename}/images"})

    if 'ImageResolutionX' in metadata.columns:
        try:
            multiscales_data[0]['axes'][3]['pixelResolution'] = float(metadata['ImageResolutionY'].iloc[0])
            multiscales_data[0]['axes'][4]['pixelResolution'] = float(metadata['ImageResolutionX'].iloc[0])
        except ValueError:
            pass

    if 'PositionZ' in metadata.columns:
        try:
            z_sorted = np.sort(metadata['PositionZ'].astype(float).unique())
            if len(z_sorted) > 1:
                mode_result = stats.mode(np.diff(z_sorted))
                if mode_result.count > 0:
                    multiscales_data[0]['axes'][2]['spacing'] = mode_result.mode
                    multiscales_data[0]['axes'][2]['unit']    = "meter"
        except ValueError:
            pass

    top_level_attrs = {
        "assay_layout": assay_layout.to_dict(orient='index'),
        "omero":        omero_metadata,
        "multiscales":  multiscales_data,
        "version":      "0.4",
    }
    group = zarr.group(zarr_root_dir, overwrite=False)
    group.attrs.put(top_level_attrs)
    print(f"OME-Zarr metadata saved: {zarr_root_dir}/.zattrs")

## 3 · Chunked tiling: Live-2
Live-2 was too large to hold in memory. This section re-processes that acquisition
by pre-allocating the full zarr array and writing one timepoint at a time directly to disk,
avoiding any in-memory accumulation of the full time series.

In [ ]:
base_dir    = '/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-2__2025-04-10T18_45_48-Measurement 1/'
base_dir_ID = 'BPP0050-1-Live-cell-to4i_Live-2__2025-04-10T18_45_48-Measurement 1'

metadata_fn = os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
metadata    = dataio.read_harmony_metadata(metadata_fn)
metadata    = metadata.apply(pd.to_numeric, errors='ignore')

# Invert Y and fix Phase Contrast Z (same preprocessing as batch loop)
metadata['PositionY'] = -metadata['PositionY']
min_z = metadata.loc[
    ~metadata['ChannelName'].str.contains('Phase Contrast', case=False, na=False), 'PositionZ'
].min()
metadata.loc[
    metadata['ChannelName'].str.contains('Phase Contrast', case=False, na=False), 'PositionZ'
] = min_z

image_dir    = os.path.join(base_dir, 'acquisition/Images/')
rows_cols    = metadata[['Row', 'Col']].drop_duplicates().values
sample_tile_id = metadata.iloc[0]

assay_layout_fn = glob.glob(os.path.join(base_dir, 'acquisition/Assaylayout/*.xml'))[0]
assay_layout    = load_assay_layout(assay_layout_fn)

for row_col in tqdm(list(reversed(rows_cols)), desc=f'Chunked tiling: {base_dir_ID}', total=len(rows_cols)):
    row, column = int(row_col[0]), int(row_col[1])
    acq_ID = (row, column)

    zarr_output_dir = Path(base_dir) / f'acquisition/zarr/{acq_ID}.zarr'
    if zarr_output_dir.exists():
        print(f"{zarr_output_dir} already exists, skipping")
        continue

    position_metadata = metadata[(metadata['Row'] == row) & (metadata['Col'] == column)]
    timepoint_ids     = sorted(position_metadata['TimepointID'].unique())

    # Pre-allocate zarr array with full known shape (T, C, Z, Y, X)
    T = len(timepoint_ids)
    C = len(position_metadata['ChannelID'].unique())
    Z = len(position_metadata['PlaneID'].unique())
    Y = (int((position_metadata['PositionY'].max() - position_metadata['PositionY'].min())
             / position_metadata['ImageResolutionY'].iloc[0])
         + position_metadata['ImageSizeY'].iloc[0])
    X = (int((position_metadata['PositionX'].max() - position_metadata['PositionX'].min())
             / position_metadata['ImageResolutionX'].iloc[0])
         + position_metadata['ImageSizeX'].iloc[0])

    root = zarr.open_group(str(zarr_output_dir), mode='w')
    zarr_array = root.create_dataset(
        name="images",
        shape=(T, C, Z, Y, X),
        chunks=(1, C, Z, Y, X),  # chunk along T for sequential write
        dtype='uint16',
        compressor=numcodecs.Blosc(cname='zstd', clevel=5, shuffle=numcodecs.Blosc.BITSHUFFLE)
    )

    for timepoint_id in tqdm(timepoint_ids, desc="Time Points", leave=True, position=1):
        time_metadata   = position_metadata[position_metadata['TimepointID'] == timepoint_id]
        channel_ids     = sorted(time_metadata['ChannelID'].unique())
        channel_volumes = []

        for channel_id in tqdm(channel_ids, desc=f"Channels (T={timepoint_id})", leave=False, position=2):
            channel_slice_metadata = time_metadata[time_metadata['ChannelID'] == channel_id]
            z_ids    = sorted(channel_slice_metadata['PositionZ'].unique())
            z_slices = []

            for z_id in z_ids:
                z_slice_metadata = channel_slice_metadata[channel_slice_metadata['PositionZ'] == z_id]
                mosaic_size_x = (
                    int((z_slice_metadata['PositionX'].max() - z_slice_metadata['PositionX'].min())
                        / sample_tile_id['ImageResolutionX'])
                    + sample_tile_id['ImageSizeX']
                )
                mosaic_size_y = (
                    int((z_slice_metadata['PositionY'].max() - z_slice_metadata['PositionY'].min())
                        / sample_tile_id['ImageResolutionY'])
                    + sample_tile_id['ImageSizeY']
                )
                accumulator = np.zeros((mosaic_size_y, mosaic_size_x), dtype=np.float32)
                weight_map  = np.zeros((mosaic_size_y, mosaic_size_x), dtype=np.float32)

                for _, tile_id in z_slice_metadata.iterrows():
                    img_path = os.path.join(image_dir, tile_id['URL'])
                    try:
                        tile = imread(img_path).astype(np.float32)
                    except FileNotFoundError:
                        tile = np.zeros((tile_id['ImageSizeY'], tile_id['ImageSizeX']), dtype=np.float32)

                    alpha_mask = create_alpha_mask(tile.shape[0], tile.shape[1], feather=0.1)
                    x_pixel = int((tile_id['PositionX'] - z_slice_metadata['PositionX'].min())
                                  / tile_id['ImageResolutionX'])
                    y_pixel = int((tile_id['PositionY'] - z_slice_metadata['PositionY'].min())
                                  / tile_id['ImageResolutionY'])

                    accumulator[y_pixel:y_pixel+tile.shape[0], x_pixel:x_pixel+tile.shape[1]] += tile * alpha_mask
                    weight_map [y_pixel:y_pixel+tile.shape[0], x_pixel:x_pixel+tile.shape[1]] += alpha_mask

                final_mosaic = np.zeros_like(accumulator)
                nonzero = weight_map > 0
                final_mosaic[nonzero] = accumulator[nonzero] / weight_map[nonzero]
                mosaic = np.clip(final_mosaic, 0, 65535).astype(np.uint16)

                if "Phase Contrast" in channel_slice_metadata['ChannelName'].iloc[0]:
                    z_slices = [mosaic] * len(metadata['PositionZ'].unique())
                    break
                else:
                    z_slices.append(mosaic)

            if z_slices:
                channel_volumes.append(np.stack(z_slices, axis=0))

        if channel_volumes:
            image_volume_czyx = np.stack(channel_volumes, axis=0)  # (C, Z, Y, X)
            zarr_array[timepoint_id] = image_volume_czyx            # write directly, no accumulation
            print(f"Timepoint {timepoint_id} saved.")